# Домашнее задание №11: адаптеры, принципы DiT и дистилляция

## Описание задачи

На лекции мы разобрали IP-Adapter и управление им по блокам (InstantStyle), ControlNet, из каких деталей собраны современные DiT-модели и как их ускоряют дистилляцией. В этом задании нужно:
- посмотреть, какие блоки UNet за что отвечают и как IP-Adapter уживается с ControlNet;
- руками собрать две детали DiT — adaLN-Zero и RoPE — и проверить их свойства тестами;
- посмотреть, что даёт и чего лишает дистилляция, на примере SDXL-Turbo.

Выбор референсных изображений — ваш. Советую взять что-то нетривиальное или близкое к сердцу (брать Ван Гога — зашквар).

**Железо.** Части 1 и 3 рассчитаны на Colab T4. Часть 2 не нужна GPU: её можно делать локально или пока качаются веса. Загрузка моделей и хелперы уже написаны — их менять не нужно. В памяти всегда одна модель: перед загрузкой следующей предыдущая выгружается. Если ловите OOM, поставьте `LOW_VRAM = True`.

---

## Система оценки

### Часть 1: Адаптеры (3 балла)

- **1.1 Какие блоки за что отвечают** (SDXL + IP-Adapter). Один референс, промпт и seed. Сравните 11 конфигураций:
  - все слои (обычный IP-Adapter);
  - отдельные transformer-блоки: `up_blocks.0.attentions.1` (стиль по InstantStyle), `down_blocks.2.attentions.1` (раскладка), любой другой блок;
  - блоки целиком: `up_blocks.0`, `down_blocks.2`, любой другой блок;
  - комбинации — хотя бы 4 на ваш выбор.

  Grid с подписями и числом слоёв в каждой конфигурации. Что переносит каждый вариант — стиль, объекты, раскладку? Почему блок целиком ближе к обычному IP-Adapter, чем один transformer-блок внутри него? — 2 балла
- **1.2 ControlNet + IP-Adapter** (SD 1.5). Структура — canny-карта, стиль — референс. Сетка `controlnet_conditioning_scale` ∈ {0.5, 1.0} × IP scale ∈ {0.3, 0.7}. Где условия начинают конфликтовать и что побеждает? — 1 балл

### Часть 2: Принципы DiT (4 балла, CPU)

Допишите места с `TODO` — под каждой функцией есть тесты.

- **2.1 adaLN-Zero.** Модуляция $\gamma, \beta, \alpha$ из вектора условия и нулевая инициализация. Почему на первом шаге градиент в attention и MLP нулевой, но обучение всё равно стартует? — 2 балла
- **2.2 RoPE с осью референса.** Поворот по осям $(k, h, w)$. Почему из поворота следует, что скор зависит только от разности позиций, и почему такая схема переносится на другое разрешение лучше абсолютных эмбеддингов? — 2 балла

### Часть 3: Дистилляция на примере SDXL-Turbo (2 балла)

SDXL-Turbo — SDXL, дистиллированный через ADD до 1–4 шагов. Учили на 512².

- **3.1 Шаги и CFG.** Turbo на 1, 2, 4 шагах без CFG: картинки и время (сравните с base из 1.1). Затем 4 шага с guidance 5 и negative prompt. Почему CFG не помогает? — 2 балла

### Итог (1 балл)

Коротко, письменно: когда что брать — IP-Adapter целиком, отдельные блоки, ControlNet, дистиллированную модель? Чем вы платите в каждом случае?

***Допбалл за кайфушность: субъективный балл за заинтересованность (необычные референсы, чувствуется любовь)***

## 0. Загрузка и хелперы (готово, менять не нужно)

Что сделано, чтобы всё влезло в T4:
- веса в fp16 (`variant="fp16"`) — иначе Colab падает по оперативке ещё при загрузке;
- VAE `madebyollin/sdxl-vae-fp16-fix` — родной VAE SDXL в fp16 даёт NaN и чёрные картинки;
- IP-Adapter-Plus с image encoder ViT-H (как на лекции);
- SDXL генерирует в 768² за 20 шагов — для сравнения блоков этого хватает, а выходит в разы быстрее, чем 1024² и 30 шагов;
- в памяти одна модель за раз: SDXL (1.1) → SD 1.5 + ControlNet (1.2) → SDXL-Turbo (часть 3).

In [ ]:
# Установка (Colab). После неё: Runtime → Restart session, затем запускайте ячейки ниже.
# torchaudio не нужен, а если он собран под другую CUDA, чем torch, — ломает импорт transformers.
!pip uninstall -y -q torchaudio
!pip install -q -U diffusers

In [ ]:
import gc
import time

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
from diffusers import AutoencoderKL, StableDiffusionXLPipeline
from diffusers.models.attention_processor import IPAdapterAttnProcessor2_0
from diffusers.utils import load_image
from PIL import Image

DEVICE = "cuda"
DTYPE = torch.float16
LOW_VRAM = False  # True — если OOM: модули по очереди переезжают на GPU (медленнее)
RES = 768         # разрешение SDXL в 1.1
STEPS = 20        # шагов SDXL в 1.1
NEG = "blurry, low quality, ugly, deformed"


def free_memory():
    gc.collect()
    torch.cuda.empty_cache()


def place(pipe):
    # offload включать только ПОСЛЕ load_ip_adapter
    if LOW_VRAM:
        pipe.enable_model_cpu_offload()
    else:
        pipe.to(DEVICE)
    return pipe


vae = AutoencoderKL.from_pretrained("madebyollin/sdxl-vae-fp16-fix", torch_dtype=DTYPE)
pipe = StableDiffusionXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    vae=vae, torch_dtype=DTYPE, variant="fp16", use_safetensors=True,
)
pipe.load_ip_adapter("h94/IP-Adapter", subfolder="sdxl_models",
                     weight_name="ip-adapter-plus_sdxl_vit-h.safetensors",
                     image_encoder_folder="models/image_encoder")  # ViT-H
pipe = place(pipe)

In [ ]:
# ── генерация и сетки ─────────────────────────────────────────────────────────
_DUMMY = Image.new("RGB", (224, 224), "gray")
_WARM = set()


def has_ip(pipe):
    return getattr(pipe.unet, "encoder_hid_proj", None) is not None


def generate(pipe, prompt, seed=42, steps=STEPS, cfg=5.0, size=RES, **kw):
    """Возвращает (картинка, секунды). Первый вызов на каждом пайплайне — прогрев, он не замеряется.
    Если IP-Adapter загружен, а картинку не передали — подставляется серая заглушка
    (без неё пайплайн падает); чтобы она не влияла, выключите адаптер: set_ip_scales(pipe, {})."""
    if has_ip(pipe) and "ip_adapter_image" not in kw:
        kw["ip_adapter_image"] = _DUMMY
    run = lambda n, g: pipe(prompt=prompt, num_inference_steps=n, guidance_scale=cfg,
                            height=size, width=size, generator=g, **kw).images[0]
    if id(pipe) not in _WARM:
        run(1, None)
        _WARM.add(id(pipe))
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    img = run(steps, torch.Generator("cpu").manual_seed(seed))
    torch.cuda.synchronize()
    return img, time.perf_counter() - t0


def show_grid(images, titles=None, ncols=None, size=3.2, suptitle=None):
    ncols = ncols or len(images)
    nrows = (len(images) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(size * ncols, size * nrows), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for i, img in enumerate(images):
        axes.flat[i].imshow(img)
        if titles:
            axes.flat[i].set_title(titles[i], fontsize=9)
    if suptitle:
        fig.suptitle(suptitle)
    plt.tight_layout()
    plt.show()


def canny(img, low=100, high=200, size=512):
    arr = np.array(img.convert("RGB").resize((size, size)))
    edges = cv2.Canny(cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY), low, high)
    return Image.fromarray(np.stack([edges] * 3, axis=-1))


# ── IP-Adapter по блокам ──────────────────────────────────────────────────────
# Конфигурация — словарь {префикс имени блока: scale}. Префикс может быть целым блоком
# ("up_blocks.0", "mid_block") или одним transformer-блоком внутри ("up_blocks.0.attentions.1").
# Слои, которые не попали ни в один префикс, получают 0.
ALL = {"down_blocks": 1.0, "mid_block": 1.0, "up_blocks": 1.0}


def _ip_layers(pipe):
    return {name: proc for name, proc in pipe.unet.attn_processors.items()
            if isinstance(proc, IPAdapterAttnProcessor2_0)}


def _match(name, cfg):
    return next((v for prefix, v in cfg.items() if name.startswith(prefix + ".")), 0.0)


def set_ip_scales(pipe, cfg):
    """Включить IP-Adapter только в слоях из cfg; set_ip_scales(pipe, {}) — выключить совсем."""
    for name, proc in _ip_layers(pipe).items():
        proc.scale = [_match(name, cfg)]


def n_ip_layers(pipe, cfg):
    return sum(_match(name, cfg) != 0 for name in _ip_layers(pipe))


print("Всего слоёв с IP-Adapter:", len(_ip_layers(pipe)))  # у SDXL — 70

### Референсы

In [ ]:
# Ваши референсы: загрузите картинки в Colab и укажите пути (или URL)
STYLE_REF = load_image("style.jpg")          # стиль
STRUCTURE_IMG = load_image("structure.jpg")  # отсюда возьмём canny для 1.2
PROMPT = "a cat sitting on a windowsill"     # можно свой
show_grid([STYLE_REF, STRUCTURE_IMG], ["стиль", "структура"])

## Часть 1: Адаптеры

### 1.1 Какие блоки за что отвечают

In [ ]:
# TODO 1.1: допишите конфигурации (всего 11, см. условие)
CONFIGS = {
    "все слои": ALL,
    "up0.att1": {"up_blocks.0.attentions.1": 1.0},
    # ...
}

images, titles = [], []
for name, cfg in CONFIGS.items():
    set_ip_scales(pipe, cfg)
    img, _ = generate(pipe, PROMPT, ip_adapter_image=STYLE_REF)
    images.append(img)
    titles.append(f"{name}\n{n_ip_layers(pipe, cfg)} слоёв")
show_grid([STYLE_REF] + images, ["референс"] + titles, ncols=4)

# время base без IP-Adapter — пригодится в 3.1
set_ip_scales(pipe, {})
_, BASE_TIME = generate(pipe, PROMPT)
print(f"base: {RES}², {STEPS} шагов, CFG — {BASE_TIME:.1f} с")

*Выводы:*

### 1.2 ControlNet + IP-Adapter (SD 1.5)

In [ ]:
# Выгружаем SDXL, грузим SD 1.5 + ControlNet canny + IP-Adapter (готово)
from diffusers import ControlNetModel, StableDiffusionControlNetPipeline

del pipe
free_memory()

controlnet = ControlNetModel.from_pretrained("lllyasviel/control_v11p_sd15_canny", torch_dtype=DTYPE,
                                             variant="fp16")
sd15 = StableDiffusionControlNetPipeline.from_pretrained(
    "stable-diffusion-v1-5/stable-diffusion-v1-5", controlnet=controlnet, torch_dtype=DTYPE, variant="fp16",
)
sd15.load_ip_adapter("h94/IP-Adapter", subfolder="models", weight_name="ip-adapter_sd15.bin")
sd15 = place(sd15)
CANNY = canny(STRUCTURE_IMG, size=512)
show_grid([STRUCTURE_IMG, CANNY], ["структура", "canny"])

In [ ]:
# TODO 1.2: сетка controlnet_conditioning_scale ∈ {0.5, 1.0} × IP scale ∈ {0.3, 0.7}
# подсказка: sd15.set_ip_adapter_scale(s); generate(sd15, PROMPT, steps=30, size=512, image=CANNY,
#            controlnet_conditioning_scale=..., ip_adapter_image=STYLE_REF)

*Выводы:*

## Часть 2: Принципы DiT (CPU)

Эту часть можно делать без GPU. Допишите места с `TODO`, затем запустите ячейку с тестами под функцией.

### 2.1 adaLN-Zero

In [ ]:
import torch
from torch import nn


class AdaLNZeroBlock(nn.Module):
    """Блок DiT: attention и MLP, условие c подаётся через adaLN-Zero.
    x: (B, N, dim) — токены, c: (B, cond_dim) — вектор условия (шаг t, guidance, ...)."""

    def __init__(self, dim, cond_dim, heads=4):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim, elementwise_affine=False, eps=1e-6)  # без своих γ, β
        self.attn = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.norm2 = nn.LayerNorm(dim, elementwise_affine=False, eps=1e-6)
        self.mlp = nn.Sequential(nn.Linear(dim, 4 * dim), nn.GELU(), nn.Linear(4 * dim, dim))
        # из c получаем 6 векторов: shift, scale, gate — для attention и для MLP
        self.modulation = nn.Sequential(nn.SiLU(), nn.Linear(cond_dim, 6 * dim))
        # TODO: adaLN-Zero — инициализируйте последний слой modulation нулями (2 строки)

    def forward(self, x, c):
        shift1, scale1, gate1, shift2, scale2, gate2 = self.modulation(c).unsqueeze(1).chunk(6, dim=-1)
        # TODO (4 строки):
        #   h = (1 + scale) * LN(x) + shift
        #   x = x + gate * Attention(h)      (self.attn(h, h, h, need_weights=False)[0])
        #   то же для MLP
        raise NotImplementedError

In [ ]:
# Тесты 2.1 (менять не нужно)
torch.manual_seed(0)
blk = AdaLNZeroBlock(dim=64, cond_dim=32)
x = torch.randn(2, 10, 64)
c = torch.randn(2, 32)
y = blk(x, c)
assert torch.allclose(y, x), "на старте блок должен быть тождественным"
(y * torch.randn_like(y)).sum().backward()
assert blk.modulation[1].weight.grad.abs().sum() > 0, "градиент по modulation должен быть ненулевым"
assert blk.attn.in_proj_weight.grad.abs().sum() == 0, "на первом шаге градиент в attention нулевой"
assert blk.mlp[0].weight.grad.abs().sum() == 0, "на первом шаге градиент в MLP нулевой"
opt = torch.optim.SGD(blk.parameters(), lr=0.1)
opt.step()
opt.zero_grad()
y = blk(x, c)
assert not torch.allclose(y, x), "после шага блок уже не тождественный"
(y * torch.randn_like(y)).sum().backward()
assert blk.attn.in_proj_weight.grad.abs().sum() > 0, "после шага градиент доходит до attention"
assert not torch.allclose(blk(x, c), blk(x, torch.randn(2, 32))), "выход должен зависеть от c"
print("2.1: все тесты пройдены")

*Почему на первом шаге градиент в attention и MLP нулевой, но обучение всё равно стартует:*

### 2.2 RoPE с осью референса

In [ ]:
def rope_1d(x, pos, theta=10000.0):
    """Поворот по одной оси. x: (N, d), d чётное; pos: (N,) — позиции токенов по этой оси.
    Пары координат (x[2i], x[2i+1]) поворачиваются на угол pos * freq_i."""
    d = x.shape[-1]
    freqs = theta ** (-torch.arange(0, d, 2, dtype=x.dtype) / d)  # (d/2,)
    ang = pos.to(x.dtype)[:, None] * freqs[None, :]                 # (N, d/2)
    cos, sin = ang.cos(), ang.sin()
    x1, x2 = x[..., 0::2], x[..., 1::2]
    # TODO: поверните каждую пару (x1, x2) на угол ang и соберите обратно в (N, d) (2 строки)
    #   (x1, x2) -> (x1 cos - x2 sin, x1 sin + x2 cos); подсказка: torch.stack(..., dim=-1).flatten(-2)
    raise NotImplementedError


def rope_nd(x, pos, axes_dims):
    """Многоосевой RoPE. x: (N, d); pos: (N, n_axes) — координаты (k, h, w);
    axes_dims — сколько каналов на каждую ось, sum(axes_dims) == d.
    Каналы делятся на куски, каждый кусок поворачивается по своей оси."""
    # TODO (2 строки): x.split(axes_dims, dim=-1) → rope_1d для каждого куска → torch.cat
    raise NotImplementedError


def make_positions(k, H, W):
    """Координаты (k, h, w) всех токенов картинки H×W с номером изображения k (готово)."""
    hh, ww = torch.meshgrid(torch.arange(H), torch.arange(W), indexing="ij")
    return torch.stack([torch.full_like(hh, k), hh, ww], dim=-1).reshape(-1, 3)

In [ ]:
# Тесты 2.2 (менять не нужно)
AXES = (16, 24, 24)  # каналы на оси (k, h, w)
D = sum(AXES)
torch.manual_seed(0)
q, k = torch.randn(1, D, dtype=torch.float64), torch.randn(1, D, dtype=torch.float64)


def score(pq, pk):
    return (rope_nd(q, torch.tensor([pq]), AXES) * rope_nd(k, torch.tensor([pk]), AXES)).sum()


X = torch.randn(5, D, dtype=torch.float64)
P = torch.randint(0, 64, (5, 3))
assert torch.allclose(rope_nd(X, P, AXES).norm(dim=-1), X.norm(dim=-1)), "поворот не меняет длину вектора"
assert torch.allclose(rope_nd(X, torch.zeros(5, 3, dtype=torch.long), AXES), X), "в нулевой позиции поворота нет"
for _ in range(5):
    pq, pk, delta = [torch.randint(0, 64, (3,)).tolist() for _ in range(3)]
    shift = lambda p: [a + b for a, b in zip(p, delta)]
    assert torch.allclose(score(pq, pk), score(shift(pq), shift(pk))), "скор должен зависеть только от разности позиций"
assert not torch.allclose(score([0, 3, 3], [0, 3, 3]), score([0, 3, 3], [0, 10, 3])), "разные смещения — разные скоры"
assert not torch.allclose(score([0, 5, 7], [0, 5, 7]), score([0, 5, 7], [1, 5, 7])), "референс отличается от картинки осью k"
assert torch.allclose(score([0, 5, 7], [1, 5, 7]), score([2, 5, 7], [3, 5, 7])), "по оси k тоже важна только разность"
assert make_positions(1, 64, 64).shape == (4096, 3)
print("2.2: все тесты пройдены")

In [ ]:
# Как скор зависит от смещения по (h, w) — для одной пары q, k (готово)
dh = dw = torch.arange(-32, 33)
S = torch.tensor([[score([0, 32, 32], [0, 32 + a, 32 + b]).item() for b in dw] for a in dh])
plt.imshow(S, extent=(-32, 32, 32, -32), cmap="RdBu_r")
plt.xlabel("Δw"); plt.ylabel("Δh"); plt.title("q·k после RoPE в зависимости от смещения"); plt.colorbar()
plt.show()

*Почему скор зависит только от разности позиций и почему это переносится на другое разрешение:*

## Часть 3: Дистилляция на примере SDXL-Turbo

In [ ]:
# Выгружаем SD 1.5, грузим SDXL-Turbo (готово). IP-Adapter здесь не нужен.
del sd15, controlnet
free_memory()

turbo = StableDiffusionXLPipeline.from_pretrained(
    "stabilityai/sdxl-turbo", vae=vae, torch_dtype=DTYPE, variant="fp16",
)
turbo = place(turbo)
TURBO_RES = 512  # Turbo учили на 512²

### 3.1 Шаги и CFG

In [ ]:
# TODO 3.1: шаги 1, 2, 4 (cfg=0, size=TURBO_RES) — картинки и время;
# затем 4 шага, cfg=5, negative_prompt=NEG. Время сравните с BASE_TIME из 1.1.

*Выводы:*

## Итог

*Ваш ответ:*